# FightWatch — RGB + tư thế + nhãn hành động phụ
Đối chứng RGB-aux và RGB-pose-aux trên split public v1, không phụ thuộc PKU. Tận dụng toàn bộ train AIRTLab/RLVS phù hợp; loại friendly-punch mơ hồ thay vì tự đổi nhãn. Chỉ train dài khi development tiến triển và không làm giảm AP nhóm khó quá 0,02. Không tự cài model/duyệt demo. Nhãn phụ là nhãn clip publisher, chưa phải temporal gold. Test v1/PKU đã được xem trước nên chỉ là chẩn đoán lặp lại.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, sys, json, shutil, subprocess, zipfile
ROOT=Path('/content/drive/MyDrive/SENTINEL')
REPO=ROOT/'repo/anomaly-behavior-detector-system'
LOCAL=Path('/content/fightwatch-fusion-code')
BASE=ROOT/'artifacts/fight_large_v1'
DATA=Path('/content/fightwatch-public-data')
PKU=Path('/content/fightwatch-fusion-pku')
CACHE=Path('/content/fightwatch-fusion-cache')
OUTPUT=ROOT/'artifacts/fight_fusion_v1'
BUDGET_MINUTES=90
VARIANT_MINUTES=25

## Đồng bộ code
Chỉ bỏ autosave notebook tracked trên bản repo Drive; dừng nếu code khác đang sửa. Không ghi đè dataset/checkpoint.

In [ ]:
subprocess.run(['git','-C',str(REPO),'restore','--','colab'],check=True)
dirty=subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True)
assert not dirty.strip(), dirty
subprocess.run(['git','-C',str(REPO),'pull','--ff-only','origin','main'],check=True)
REVISION=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
shutil.copytree(REPO,LOCAL,dirs_exist_ok=True,ignore=shutil.ignore_patterns('.git','.venv','logs','uploads','models','__pycache__','.pytest_cache'))
os.chdir(LOCAL)
from scripts.colab_process import run_logged
print('Code:',REVISION)

In [ ]:
if not (OUTPUT/'run.json').exists():
    run_logged([sys.executable,'-m','pip','install','-q','-r','requirements-colab.txt'],OUTPUT/'install.log',900,revision=REVISION)
    import torch
    assert torch.cuda.is_available(), 'Chọn GPU trước khi train'
    print('GPU:',torch.cuda.get_device_name(0))
    run_logged([sys.executable,'-m','scripts.fight_public_data','--archives',str(ROOT/'datasets/fight'),'--data',str(DATA),'--output',str(BASE)],OUTPUT/'data.log',1800,revision=REVISION)
    source=ROOT/'datasets/fight (1)/pku_mmd_sample'
    if source.is_dir():
        (PKU/'rgb').mkdir(parents=True,exist_ok=True)
        (PKU/'labels').mkdir(parents=True,exist_ok=True)
        for video in sorted((source/'rgb').glob('*.avi')):
            labels=list((source/'labels').rglob(video.stem+'.txt'))
            assert len(labels)==1,video.name
            shutil.copyfile(video,PKU/'rgb'/video.name)
            shutil.copyfile(labels[0],PKU/'labels'/labels[0].name)
        shutil.copyfile(source/'Actions.xlsx',PKU/'Actions.xlsx')
else:
    print('Đã có lượt chạy; chỉ đọc báo cáo, không tự train lại.')

## Training có điều kiện tiếp tục
Mỗi cấu hình 3 probe + tối đa 18 fine-tune, tối đa 25 phút/variant. Sau 3 epoch fine-tune, chỉ tiếp tục khi worst-cohort precision/recall tăng ≥0,01 hoặc macro AP tăng ≥0,005 và AP từng nguồn không giảm quá 0,02. Patience 4. Toàn cache/train/eval/export tối đa 90 phút, watchdog 92 phút. Gate chọn trên development whole-clip any-alarm, không gọi đó là độ chính xác mốc sự kiện. PKU chỉ chấm sau đóng băng, không train/chọn model/ngưỡng/gate.

In [ ]:
if not (OUTPUT/'run.json').exists():
    command=[sys.executable,'-m','scripts.fight_fusion_train','--baseline',str(BASE),'--data',str(DATA),'--cache',str(CACHE),'--output',str(OUTPUT),'--budget-minutes',str(BUDGET_MINUTES),'--variant-minutes',str(VARIANT_MINUTES)]
    if (PKU/'Actions.xlsx').is_file():
        command+=['--pku',str(PKU)]
    run_logged(command,OUTPUT/'training.log',(BUDGET_MINUTES+2)*60,revision=REVISION)
else:
    print('Không tự retry một run đã bắt đầu.')

In [ ]:
report=json.loads((OUTPUT/'run.json').read_text())
print('STATUS:',report['status'],'seconds:',report.get('seconds'))
print('Counts:',report.get('counts'))
for name,variant in report['variants'].items():
    print(name,'epochs:',variant['completed_epochs'],'stop:',variant['stop_reason'])
    print('Public clip diagnostics:',variant['test_by_dataset'])
    print('Hard negatives:',variant['hard_negative_actions'])
    print('Event gate:',variant['event_gate'])
    for source in variant.get('pku_continuous',[]):
        print(source['video'],source['metrics'])
print('No physical iPhone tested; no automatic deployment.')
archive_path=OUTPUT/'window_cache.zip'
if CACHE.exists() and not archive_path.exists():
    partial=archive_path.with_suffix('.partial.zip')
    with zipfile.ZipFile(partial,'w',zipfile.ZIP_STORED) as archive:
        for cached in sorted(CACHE.glob('*.npz')):
            archive.write(cached,cached.name)
    partial.replace(archive_path)
print('Saved:',OUTPUT)

In [ ]:
render=[sys.executable,'-m','scripts.fight_fusion_report',str(OUTPUT)]
if (PKU/'Actions.xlsx').is_file():
    render+=['--pku',str(PKU)]
run_logged(render,OUTPUT/'render.log',120,revision=REVISION)
from IPython.display import Image, display
for image in sorted(OUTPUT.glob('*/learning_curves.png')):
    display(Image(filename=str(image)))
print('Paired camera views are grouped in grouped_hard_negatives.json; no physical iPhone tested.')

In [ ]:
qa=[sys.executable,'-m','scripts.fight_fusion_qa','pack',str(OUTPUT),str(CACHE)]
if (PKU/'Actions.xlsx').is_file():
    qa+=['--pku',str(PKU)]
run_logged(qa,OUTPUT/'qa_package.log',180,revision=REVISION)
print('CPU QA:',OUTPUT/'cpu_validation.zip')
print('Runtime từng variant có model, metadata và pose weights; chưa tự cài vào local.')